# Function calling: el contrato entre el modelo y tus herramientas

**Facsímil 5 · Agentes y orquestación** — capítulo 3 (*tools* y contratos operativos: *function
calling*).

Entre el modelo y tu base de datos (o tu cartera) hay una **frontera**. El modelo no «ejecuta» tus
funciones: **pide** ejecutarlas rellenando un formulario (nombre de la herramienta + argumentos), y tu
código decide si lo concede. *Function calling* es esa frontera, y es tan importante como lo fue la
separación entre datos y comandos para frenar la inyección SQL. En este cuaderno defines herramientas
con su **contrato**, aceptas una llamada bien hecha y **rechazas** una mal hecha *antes* de que cause
daño. Sin validación, una alucinación del modelo se convierte en una acción real.

### Qué vas a aprender
- Que el modelo **propone** una llamada (herramienta + argumentos) y tu código **dispone**.
- A declarar el **contrato** de cada herramienta (qué argumentos acepta y de qué tipo) con Pydantic.
- A **validar** una llamada y rechazar las mal formadas con un motivo claro, sin tocar nada.
- A poner **permisos**: qué acciones puede ejecutar el agente solo y cuáles requieren confirmación.

### Cuánto cuesta
Unos 12 minutos. CPU, sin claves (simulamos lo que pide el modelo).


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [1]:
# En Colab:  !pip install pydantic
from pydantic import BaseModel, field_validator, ValidationError

# Cada herramienta declara su CONTRATO: que argumentos acepta y de que tipo.
class Convertir(BaseModel):
    cantidad: float
    de: str
    a: str
    @field_validator("de", "a")
    @classmethod
    def moneda_valida(cls, v):
        if v.upper() not in {"EUR", "USD", "GBP"}:
            raise ValueError(f"moneda no soportada: {v}")
        return v.upper()

class Recordatorio(BaseModel):
    texto: str
    minutos: int
    @field_validator("minutos")
    @classmethod
    def futuro(cls, v):
        if v <= 0: raise ValueError("los minutos deben ser positivos")
        return v

CONTRATOS = {"convertir": Convertir, "recordatorio": Recordatorio}
TASAS = {("USD","EUR"): 0.92, ("EUR","USD"): 1.09, ("GBP","EUR"): 1.17}
print("Herramientas con contrato:", ", ".join(CONTRATOS))


Herramientas con contrato: convertir, recordatorio


## 1. El modelo propone; tú dispones

Cuando un LLM «usa una herramienta», en realidad devuelve un objeto: el **nombre** de la herramienta y
unos **argumentos** en JSON. No lo ejecutamos a ciegas. Lo pasamos por el **contrato** (el esquema de
esa herramienta). Si encaja, se ejecuta; si no, se rechaza con un motivo claro, **sin tocar nada**. Esta
separación —proponer vs. ejecutar— es lo que mantiene el control en tu lado, no en el del modelo.


In [2]:
def ejecutar(nombre, args):
    if nombre == "convertir":
        t = TASAS.get((args.de, args.a))
        return f"{args.cantidad} {args.de} = {round(args.cantidad*t, 2)} {args.a}" if t else "par no disponible"
    if nombre == "recordatorio":
        return f"recordatorio '{args.texto}' en {args.minutos} min: creado"

def maneja(llamada):
    nombre, args_json = llamada["tool"], llamada["args"]
    if nombre not in CONTRATOS:
        return f"[RECHAZADA] herramienta desconocida: {nombre}"
    try:
        args = CONTRATOS[nombre].model_validate(args_json)   # valida contra el contrato
    except ValidationError as e:
        return f"[RECHAZADA] {nombre}: {e.errors()[0]['loc'][0]} -> {e.errors()[0]['msg']}"
    return f"[OK] {ejecutar(nombre, args)}"

print("Llamadas BIEN hechas:")
for ll in [{"tool":"convertir","args":{"cantidad":100,"de":"usd","a":"eur"}},
           {"tool":"recordatorio","args":{"texto":"llamar al banco","minutos":30}}]:
    print(" ", maneja(ll))


Llamadas BIEN hechas:
  [OK] 100.0 USD = 92.0 EUR
  [OK] recordatorio 'llamar al banco' en 30 min: creado


## 2. Cuando el modelo alucina los argumentos

Los modelos, a veces, inventan una moneda, ponen un texto donde va un número o se dejan un campo. Sin
contrato, eso llega a tu lógica y revienta (o peor, hace algo). Con contrato, **se queda en la puerta**
con un motivo legible, antes de ejecutarse. Probamos cuatro alucinaciones típicas.


In [3]:
malas = [
    ("moneda inventada",      {"tool":"convertir","args":{"cantidad":100,"de":"BTC","a":"eur"}}),
    ("minutos negativos",     {"tool":"recordatorio","args":{"texto":"x","minutos":-5}}),
    ("falta un campo",        {"tool":"recordatorio","args":{"texto":"falta el tiempo"}}),
    ("herramienta inexistente", {"tool":"borrar_todo","args":{}}),
]
for motivo, ll in malas:
    print(f"  {motivo:<22} -> {maneja(ll)}")


  moneda inventada       -> [RECHAZADA] convertir: de -> Value error, moneda no soportada: BTC
  minutos negativos      -> [RECHAZADA] recordatorio: minutos -> Value error, los minutos deben ser positivos
  falta un campo         -> [RECHAZADA] recordatorio: minutos -> Field required
  herramienta inexistente -> [RECHAZADA] herramienta desconocida: borrar_todo


**Eso es el valor del contrato.** Cada llamada peligrosa se ha quedado en la puerta, con un motivo
legible, *antes* de ejecutarse. La moneda inventada, el tiempo imposible, el campo que falta y la
herramienta que no existe: ninguna ha llegado a tocar nada. El modelo propone; tu código es quien
decide. Es exactamente la misma idea que las salidas estructuradas del facsímil 4, aplicada a las
acciones de un agente.


## 3. Permisos: no todo debe ser automático

Validar que una llamada está **bien formada** no es lo mismo que decidir que se puede **ejecutar sin
supervisión**. Convertir monedas es inofensivo; *enviar dinero* o *borrar una cuenta* no. Por eso los
agentes serios clasifican las herramientas por **riesgo** y exigen confirmación humana para las
peligrosas (capítulo 8). Añadimos una herramienta sensible y una capa de permisos.


In [4]:
HERRAMIENTAS_SENSIBLES = {"enviar_dinero", "borrar_cuenta"}

def maneja_con_permisos(llamada, confirmado=False):
    nombre = llamada["tool"]
    if nombre in HERRAMIENTAS_SENSIBLES and not confirmado:
        return f"[PENDIENTE] '{nombre}' es una accion sensible: requiere confirmacion humana."
    if nombre in HERRAMIENTAS_SENSIBLES and confirmado:
        return f"[OK] '{nombre}' ejecutada (un humano lo autorizo)."
    return maneja(llamada)

env = {"tool":"enviar_dinero","args":{}}
print("Sin confirmar:", maneja_con_permisos(env, confirmado=False))
print("Confirmado:   ", maneja_con_permisos(env, confirmado=True))
print("Inofensiva:   ", maneja_con_permisos({"tool":"convertir","args":{"cantidad":50,"de":"eur","a":"usd"}}))
print("\nEl modelo puede PEDIR cualquier accion; los permisos deciden cual se ejecuta sola y cual no.")


Sin confirmar: [PENDIENTE] 'enviar_dinero' es una accion sensible: requiere confirmacion humana.
Confirmado:    [OK] 'enviar_dinero' ejecutada (un humano lo autorizo).
Inofensiva:    [OK] 50.0 EUR = 54.5 USD

El modelo puede PEDIR cualquier accion; los permisos deciden cual se ejecuta sola y cual no.


## 4. Pruébalo tú

1. **Añade un límite de cantidad** en `Convertir` (máximo 10.000). Una llamada por un millón debería
   rechazarse: los contratos también ponen topes de negocio, no solo de tipo.
2. **Argumentos de más:** manda un campo extra que el contrato no espera. ¿Lo ignora o lo rechaza? Pydantic
   te deja elegir la política (`model_config`).
3. **Una herramienta nueva sensible** (`cambiar_contrasena`) y haz que pase por la capa de permisos.
4. **Registra cada llamada** (aceptada o rechazada) en una lista: eso es la auditoría de un agente, la
   base para entender qué intentó hacer.


## 5. Errores comunes

- **Ejecutar lo que el modelo pide sin validar.** Una alucinación se convierte en una acción real. Valida
  *siempre* contra un contrato.
- **Confundir «bien formado» con «permitido».** Una llamada puede ser válida y aun así requerir
  confirmación humana. Tipos y permisos son capas distintas.
- **Mensajes de error opacos.** Si el rechazo no dice qué falló, ni tú ni el modelo (en el reintento)
  podéis corregirlo.
- **No auditar.** Sin registro de qué herramientas pidió el agente, es imposible entender un incidente.


## 6. Qué te llevas

- *Function calling* es un **contrato**: el modelo rellena un formulario (herramienta + argumentos) y tu
  código lo **valida antes de ejecutar**.
- La validación convierte una posible alucinación en un **rechazo limpio**, no en una acción real con
  consecuencias.
- Esta frontera es donde se ponen los **permisos**: qué puede hacer el agente solo y qué necesita la
  autorización de una persona.

**Para seguir:** el capítulo 8 trata permisos y autonomía a fondo; el capítulo 10, cómo evaluar si el
agente, con todas sus herramientas, hace bien su trabajo; y el facsímil 9 conecta esto con la seguridad
(un texto externo que intenta forzar una llamada peligrosa: inyección de prompt).


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*